# YOLO Dataset Creation & Fine-Tuning

**Architecture: 2-class model** — YOLO detects `person` (0) and `ball` (1) only.  
Role assignment (team A/B, goalkeeper, referee) is handled in post-processing via KMeans clustering on ResNet18 appearance embeddings.

**Phase 1** — Remap original 300 Budućnost–Sutjeska labels (4-class → 2-class) and retrain YOLO from scratch.  
**Phase 2** — Extract 300 frames from 5 additional games, auto-label persons, annotate ball in CVAT, then fine-tune the 2-class model.

In [1]:
import sys
import importlib
import torch
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import src.dataset
importlib.reload(src.dataset)

from src.config import Config
from src.dataset import create_data_yaml, split_val, visualize_labels

dataset_dir = Config.PROJECT_ROOT / "dataset"
train_imgs = len(list((dataset_dir / "images" / "train").glob("*.jpg")))
val_imgs = len(list((dataset_dir / "images" / "val").glob("*.jpg"))) if (dataset_dir / "images" / "val").exists() else 0
print(f"Dataset: {dataset_dir}")
print(f"  Total: {train_imgs + val_imgs} frames ({train_imgs} train, {val_imgs} val)")
if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    device_arch = f"sm_{cc_major}{cc_minor}"
    supported_arches = set(torch.cuda.get_arch_list())
    gpu_name = torch.cuda.get_device_name(0)
    print(f"CUDA: enabled ({gpu_name})")
    if device_arch in supported_arches:
        print(f"PyTorch CUDA arch support: OK ({device_arch})")
    else:
        print(f"PyTorch CUDA arch support: MISSING ({device_arch})")
        print("Install a newer PyTorch build (CUDA 12.8/13.0) to use GPU for training.")
else:
    print("CUDA: not available (training will use CPU)")

Dataset: C:\Users\PC\Desktop\GitHub\football-computer-vision\dataset
  Total: 300 frames (240 train, 60 val)
CUDA: enabled (NVIDIA GeForce RTX 5070)
PyTorch CUDA arch support: OK (sm_120)


## Step 1 — Remap Existing Labels to 2-Class

Before retraining, convert all existing labels in `dataset/labels/` from 4-class to 2-class:

Run the cell below, then retrain YOLO from scratch on the remapped labels.

In [2]:
from pathlib import Path

dataset_dir = Path("dataset")

# Old class ID → new class ID
REMAP = {0: 0, 1: 1, 2: 0, 3: 0}   # goalkeeper(2) + referee(3) → person(0)

modified = 0
unchanged = 0
total_boxes = 0

for split in ["train", "val"]:
    lbl_dir = dataset_dir / "labels" / split
    if not lbl_dir.exists():
        continue
    for txt in sorted(lbl_dir.glob("*.txt")):
        raw = txt.read_text().strip()
        if not raw:
            continue
        original_lines = raw.splitlines()
        new_lines = []
        changed = False
        for line in original_lines:
            parts = line.split()
            if not parts:
                continue
            old_cls = int(parts[0])
            new_cls = REMAP.get(old_cls, old_cls)
            if new_cls != old_cls:
                changed = True
            new_lines.append(f"{new_cls} " + " ".join(parts[1:]))
        txt.write_text("\n".join(new_lines))
        total_boxes += len(new_lines)
        if changed:
            modified += 1
        else:
            unchanged += 1

print(f"Remapped  : {modified} files updated  ({unchanged} already correct)")
print(f"Boxes kept: {total_boxes}")
print("Classes are now: 0 = person  (was player / goalkeeper / referee)")
print("                 1 = ball")

Remapped  : 0 files updated  (275 already correct)
Boxes kept: 3257
Classes are now: 0 = person  (was player / goalkeeper / referee)
                 1 = ball


In [3]:
# Step 2 — Retrain YOLO from scratch (2-class: person + ball)
# After running the remap cell above, un-comment and run this cell.

import os, importlib
import torch
from pathlib import Path
from ultralytics import YOLO

import src.dataset
importlib.reload(src.dataset)
from src.config import Config
from src.dataset import create_data_yaml, split_val

dataset_dir = Config.PROJECT_ROOT / "dataset"

# Re-write data.yaml to 2 classes and re-split val
create_data_yaml(dataset_dir)

force_resplit = False
val_dir = dataset_dir / "images" / "val"
val_count = len(list(val_dir.glob("*.jpg"))) if val_dir.exists() else 0
if force_resplit or val_count == 0:
    split_val(dataset_dir, val_ratio=0.2)
else:
    print(f"Skipping split_val (already have {val_count} val images).")

# Delete stale label cache files — they may contain class IDs from a previous
# training run (e.g. 4-class model) and will cause a CUDA device-side assert
# when training with a different number of classes.
for cache in (dataset_dir / "labels").rglob("*.cache"):
    cache.unlink()
    print(f"Deleted stale cache: {cache.name}")

torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

use_cuda = False
if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    if f"sm_{cc_major}{cc_minor}" in set(torch.cuda.get_arch_list()):
        use_cuda = True
    else:
        print(f"GPU detected but PyTorch does not support sm_{cc_major}{cc_minor}. Falling back to CPU.")
device  = 0 if use_cuda else "cpu"
workers = max(2, min(8, os.cpu_count() or 2))
# Use a fixed batch size; batch=-1 (auto-detect) can corrupt GPU memory state
# on new architectures and cause silent CUDA errors during training.
batch   = 8 if use_cuda else 16

# Train from the base YOLOv8n checkpoint (2-class from scratch)
model = YOLO("yolov8n.pt")
model.train(
    data          = str(dataset_dir / "data.yaml"),
    epochs        = 200,
    imgsz         = 1280,
    batch         = batch,
    device        = device,
    workers       = workers,
    cache         = True,
    amp           = use_cuda,
    deterministic = False,
    name          = "football_2class_v1",
    patience      = 20,
    copy_paste    = 0.3,
)


Created C:\Users\PC\Desktop\GitHub\football-computer-vision\dataset\data.yaml
Skipping split_val (already have 60 val images).
New https://pypi.org/project/ultralytics/8.4.27 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.14.3 torch-2.12.0.dev20260324+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\PC\Desktop\GitHub\football-computer-vision\dataset\data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, ke

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002AAD43DFAF0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

## Ball Detection — Additional Annotation Data for CVAT

**Goal**: Sample random frames from multiple matches, auto-label person with the existing model, then package everything as a single **CVAT YOLO 1.1** zip so you only draw ball boxes.

---

### Your job in CVAT (per frame)
- **Add** ball bounding boxes — the whole point
- **Delete** obvious false positives (boxes on flags, ad boards, crowd)
- **Add** any clearly missed players near the ball
- **Skip** borderline cases (heavily occluded / blurry players at edge)

---

### End-to-end workflow
1. Add videos to `videos/` and register them in `Config.BALL_ANNOTATION_VIDEOS`
2. Run the three cells below → produces `ball_annotation_cvat.zip`
3. **CVAT**: *Create Task* → upload images → *Upload Annotations → YOLO 1.1*
4. Draw ball boxes + fix obvious errors
5. Export → *YOLO 1.1* → rename zip to `ball_annotation_cvat_annotated.zip`
6. Run the **Merge & Fine-tune** cells at the bottom

In [4]:
import sys, cv2, random, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import src.config
importlib.reload(src.config)

from src.config import Config
from ultralytics import YOLO

# ── Parameters ──────────────────────────────────────────────────────────────
CONF_THRESH    = 0.30      # confidence for auto-labeling persons
AVOID_ENDS_SEC = 300        # skip first/last N seconds (intro / outro)

# 2-class schema — matches the trained model and the training dataset exactly
CLASS_NAMES    = ["person", "ball"]
AUTO_LABEL_CLS = {0}       # person only — ball (1) is left empty for CVAT

WORK_DIR = Config.PROJECT_ROOT / "ball_annotation_work"
ZIP_OUT  = Config.PROJECT_ROOT / "ball_annotation_cvat.zip"
# ────────────────────────────────────────────────────────────────────────────

# Load the current best fine-tuned model
model_path = Config.resolve_yolo_model()
print(f"Using model : {model_path}")
model = YOLO(model_path)

# Resolve and validate video list
video_entries = Config.BALL_ANNOTATION_VIDEOS
print(f"\nVideos configured : {len(video_entries)}")
total_planned = 0
for entry in video_entries:
    vp = Path(entry["path"])
    status = "OK" if vp.exists() else "MISSING"
    print(f"  [{status}] {vp.name}  (slug={entry['slug']}, n={entry['n_frames']})")
    if vp.exists():
        total_planned += entry["n_frames"]

print(f"\nTotal frames to extract : {total_planned}")


Using model : C:\Users\PC\Desktop\GitHub\football-computer-vision\runs\detect\football_2class_v12\weights\best.pt

Videos configured : 5
  [OK] ARSENAL - DECIC 1.CFL 21.03.2026. CIJELA UTAKMICA.mp4  (slug=ars-dec, n=60)
  [OK] BOKELJ - JEDINSTVO 1.CFL 22.KOLO 01.03.2026. CIJELA UTAKMICA.mp4  (slug=bok-jed, n=60)
  [OK] JEZERO - JEDINSTVO 1.CFL 21.03.2026. CIJELA UTAKMICA.mp4  (slug=jez-jed, n=60)
  [OK] PETROVAC - MORNAR 1.CFL 21.03.2026. CIJELA UTAKMICA.mp4  (slug=pet-mor, n=60)
  [OK] SUTJESKA - MLADOST 1.CFL 21.03.2026. CIJELA UTAKMICA.mp4  (slug=sut-mla, n=60)

Total frames to extract : 300


In [5]:
from tqdm import tqdm


def stratified_sample(start: int, end: int, n: int, seed: int) -> list[int]:
    """
    Divide [start, end) into n equal segments and pick one random frame
    from each segment.  Guarantees even coverage across the entire match.
    """
    rng      = random.Random(seed)
    size     = (end - start) / n
    selected = []
    for i in range(n):
        seg_start = int(start + i * size)
        seg_end   = max(seg_start + 1, int(start + (i + 1) * size))
        selected.append(rng.randint(seg_start, seg_end - 1))
    return selected


WORK_DIR.mkdir(parents=True, exist_ok=True)
img_dir = WORK_DIR / "obj_train_data"
img_dir.mkdir(exist_ok=True)

all_frame_names = []

for entry in video_entries:
    video_path = Path(entry["path"])
    slug       = entry["slug"]
    n_frames   = entry["n_frames"]
    seed       = entry["seed"]

    if not video_path.exists():
        print(f"SKIP (missing): {video_path.name}")
        continue

    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)

    start    = int(fps * AVOID_ENDS_SEC)
    end      = total - int(fps * AVOID_ENDS_SEC)
    selected = stratified_sample(start, end, min(n_frames, end - start), seed)

    print(f"\n{video_path.name}")
    print(f"  {total / fps / 60:.1f} min  |  sampling {len(selected)} frames "
          f"(1 per {(end - start) / len(selected) / fps:.0f}s segment)")

    skipped = 0
    for frame_num in tqdm(selected, desc=f"  {slug}"):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        if not ret:
            skipped += 1
            continue

        h, w  = frame.shape[:2]
        fname = f"{slug}_frame_{frame_num:07d}"
        all_frame_names.append(fname)

        # ── Save JPEG ────────────────────────────────────────────────────────
        cv2.imwrite(str(img_dir / f"{fname}.jpg"), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])

        # ── Auto-label player / goalkeeper / referee (ball left empty) ───────
        results = model(frame, conf=CONF_THRESH, verbose=False)[0]
        lines   = []
        for box in results.boxes:
            cls_id = int(box.cls)
            if cls_id in AUTO_LABEL_CLS:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx = (x1 + x2) / 2 / w
                cy = (y1 + y2) / 2 / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h
                lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        (img_dir / f"{fname}.txt").write_text("\n".join(lines))

    cap.release()
    if skipped:
        print(f"  ({skipped} frames unreadable, skipped)")

print(f"\nTotal extracted : {len(all_frame_names)} frames  →  {img_dir}")


ARSENAL - DECIC 1.CFL 21.03.2026. CIJELA UTAKMICA.mp4
  99.1 min  |  sampling 60 frames (1 per 89s segment)


  ars-dec: 100%|██████████| 60/60 [00:31<00:00,  1.90it/s]



BOKELJ - JEDINSTVO 1.CFL 22.KOLO 01.03.2026. CIJELA UTAKMICA.mp4
  112.4 min  |  sampling 60 frames (1 per 102s segment)


  bok-jed: 100%|██████████| 60/60 [00:02<00:00, 20.45it/s]



JEZERO - JEDINSTVO 1.CFL 21.03.2026. CIJELA UTAKMICA.mp4
  107.0 min  |  sampling 60 frames (1 per 97s segment)


...


PETROVAC - MORNAR 1.CFL 21.03.2026. CIJELA UTAKMICA.mp4
  109.1 min  |  sampling 60 frames (1 per 99s segment)


  pet-mor: 100%|██████████| 60/60 [00:03<00:00, 19.15it/s]



SUTJESKA - MLADOST 1.CFL 21.03.2026. CIJELA UTAKMICA.mp4
  104.1 min  |  sampling 60 frames (1 per 94s segment)


  sut-mla: 100%|██████████| 60/60 [00:02<00:00, 21.36it/s]


Total extracted : 300 frames  →  C:\Users\PC\Desktop\GitHub\football-computer-vision\ball_annotation_work\obj_train_data


In [ ]:
import zipfile

# ── CVAT YOLO 1.1 metadata files ────────────────────────────────────────────
(WORK_DIR / "obj.names").write_text("\n".join(CLASS_NAMES))

(WORK_DIR / "obj.data").write_text(
    f"classes = {len(CLASS_NAMES)}\n"
    f"train  = train.txt\n"
    f"names  = obj.names\n"
    f"backup = backup/\n"
)

(WORK_DIR / "train.txt").write_text(
    "\n".join(f"obj_train_data/{n}.jpg" for n in all_frame_names)
)

# ── Zip everything ───────────────────────────────────────────────────────────
with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in WORK_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(WORK_DIR))

size_mb = ZIP_OUT.stat().st_size / 1e6
print(f"CVAT package : {ZIP_OUT.name}  ({size_mb:.1f} MB)")
print(f"  {len(all_frame_names)} images")
print(f"  person boxes pre-annotated")
print(f"  ball boxes left EMPTY (your job in CVAT)")
print()
print("── How to import into CVAT ──────────────────────────────────────────")
print("  1. Create a task, upload the images from this zip")
print("  2. 'Upload Annotations' → choose format 'YOLO 1.1' → upload same zip")
print("  3. Draw BALL boxes; fix obvious wrong player boxes; skip borderline cases")
print("  4. Export task as 'YOLO 1.1' → download zip")
print("  5. Rename to ball_annotation_cvat_annotated.zip")
print("  6. Run the 'Merge & Fine-tune' cells below")

CVAT package : ball_annotation_cvat.zip  (122.6 MB)
  300 images
  player / goalkeeper / referee boxes pre-annotated
  ball boxes left EMPTY (your job in CVAT)

── How to import into CVAT ──────────────────────────────────────────
  1. Create a task, upload the images from this zip
  2. 'Upload Annotations' → choose format 'YOLO 1.1' → upload same zip
  3. Draw BALL boxes; fix obvious wrong player boxes; skip borderline cases
  4. Export task as 'YOLO 1.1' → download zip
  5. Rename to ball_annotation_cvat_annotated.zip
  6. Run the 'Merge & Fine-tune' cells below


## After CVAT Annotation — Merge & Fine-tune

After exporting the annotated task from CVAT as **YOLO 1.1**, rename it to `ball_annotation_cvat_annotated.zip`, place it in the project root, then run the two cells below.

### What the merge does
- Copies every image + label from the CVAT zip into `dataset/images/train/` and `dataset/labels/train/`
- Labels use the same 2-class schema as the training dataset (`person=0`, `ball=1`) — no remapping needed
- A short fine-tune from the existing `best.pt` reinforces ball detection without forgetting person detection

### Fine-tune settings for small-object ball detection
| Setting | Rationale |
|---|---|
| `imgsz=1280` | Ball is ~15 px at 720p; doubling resolution makes it ~30 px |
| `epochs=50` | Short fine-tune from checkpoint — no need to re-learn everything |
| `lr0=5e-4` | Low LR to preserve existing person detection weights |
| `mosaic=1.0` | Mixes 4 frames per sample — ball more likely to appear |
| `copy_paste=0.3` | Pastes ball crops into other frames → implicit oversampling |


In [2]:
import sys, zipfile, shutil
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from src.config import Config

# ── Point this at the CVAT export zip ───────────────────────────────────────
CVAT_EXPORT_ZIP = Config.PROJECT_ROOT / "ball_annotation_cvat_annotated.zip"
# ────────────────────────────────────────────────────────────────────────────

# CVAT YOLO 1.1 exports contain only label files — images are NOT included.
# The original frames were saved here during extraction (cell 8):
FRAMES_DIR = Config.PROJECT_ROOT / "ball_annotation_work" / "obj_train_data"

dataset_dir = Config.PROJECT_ROOT / "dataset"
img_out_dir = dataset_dir / "images" / "train"
lbl_out_dir = dataset_dir / "labels" / "train"

if not CVAT_EXPORT_ZIP.exists():
    raise FileNotFoundError(f"Place your CVAT export at:\n  {CVAT_EXPORT_ZIP}")

added_images = 0
added_labels = 0
missing_images = []

with zipfile.ZipFile(CVAT_EXPORT_ZIP) as zf:
    for name in zf.namelist():
        p = Path(name)
        # CVAT YOLO 1.1 puts label files inside obj_train_data/
        if p.parent.name != "obj_train_data" or p.suffix != ".txt":
            continue

        # Copy label from zip
        raw = zf.read(name).decode("utf-8")
        (lbl_out_dir / p.name).write_text(raw)
        added_labels += 1

        # Copy matching image from the local frames directory
        img_src = FRAMES_DIR / p.with_suffix(".jpg").name
        if img_src.exists():
            shutil.copy2(img_src, img_out_dir / img_src.name)
            added_images += 1
        else:
            missing_images.append(img_src.name)

print(f"Merged  : {added_images} images, {added_labels} label files → dataset/train/")
if missing_images:
    print(f"WARNING : {len(missing_images)} images not found in {FRAMES_DIR}")
    for m in missing_images[:5]:
        print(f"  missing: {m}")
print(f"Total train images now: {len(list(img_out_dir.glob('*.jpg')))}")


Merged  : 300 images, 300 label files → dataset/train/
Total train images now: 540


In [3]:
import os, torch
from ultralytics import YOLO
from src.config import Config
from src.dataset import split_val, create_data_yaml

dataset_dir = Config.PROJECT_ROOT / "dataset"

# Re-create data.yaml and re-split val with the enlarged dataset.
# split_val merges any existing val/ back into train/ first, so all 600
# frames are pooled before the 80/20 split → 480 train, 120 val.
create_data_yaml(dataset_dir)
split_val(dataset_dir, val_ratio=0.2)

# Delete stale label cache files before training to avoid CUDA class-index errors
for cache in (dataset_dir / "labels").rglob("*.cache"):
    cache.unlink()
    print(f"Deleted stale cache: {cache.name}")

# ── Training setup ───────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

use_cuda = False
if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    if f"sm_{cc_major}{cc_minor}" in set(torch.cuda.get_arch_list()):
        use_cuda = True
device  = 0 if use_cuda else "cpu"
workers = max(2, min(8, os.cpu_count() or 2))
# Use a fixed batch size; batch=-1 (auto-detect) can corrupt GPU memory state
# on new architectures and cause silent CUDA errors during training.
batch   = 4 if use_cuda else 4

# Start from the existing best checkpoint
best_ckpt = Config.resolve_yolo_model()
print(f"Fine-tuning from : {best_ckpt}")
print(f"Device           : {'CUDA' if use_cuda else 'CPU'}")

model = YOLO(best_ckpt)
model.train(
    data          = str(dataset_dir / "data.yaml"),
    epochs        = 50,
    imgsz         = 1280,     # higher res → ball is larger → easier to detect
    batch         = batch,
    device        = device,
    workers       = workers,
    lr0           = 5e-4,     # low LR — preserve existing weights
    lrf           = 0.01,
    mosaic        = 1.0,
    copy_paste    = 0.3,      # paste ball crops into other frames
    cache         = True,
    amp           = use_cuda,
    deterministic = False,
    name          = "football_ball_finetune_v1",
    patience      = 15,
)


Created C:\Users\PC\Desktop\GitHub\football-computer-vision\dataset\data.yaml
Split: 510 train, 90 val (from 600 total, 15% split)
Deleted stale cache: train.cache
Deleted stale cache: val.cache
Fine-tuning from : C:\Users\PC\Desktop\GitHub\football-computer-vision\runs\detect\football_2class_v12\weights\best.pt
Device           : CUDA
New https://pypi.org/project/ultralytics/8.4.27 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.14.3 torch-2.12.0.dev20260324+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\PC\Desktop\GitHub\football-computer-vision\dataset\data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=Non

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000027F92CE4360>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0